# 02 · Pilot: is the scorer the generator? (design doc §7.6)

**How to run**: pick a Colab server with a GPU (a T4 is enough) and click "Run All"; about 30 minutes. Results go to `DATA_DIR/02_pilot_scorer_family/`.

**What it computes**: no new text is generated. The continuations of notebook 01 are rescored by the model that wrote them, and Fast-DetectGPT is computed again.

With GPT-2 XL as the scorer, the pure-sampled texts of notebook 01 show a reversed direction (AUROC 0.29 to 0.53). Dubois et al. (EMNLP 2025 Findings) found no reversal for pure sampling when the scorer is from the same family. This experiment tests whether the reversal requires a scorer that differs from the generator.

In [ ]:
# 1. Check for a GPU. The largest model is 3B, so a T4 is enough.
import torch
assert torch.cuda.is_available(), "No GPU: switch the Colab server to a GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")

In [ ]:
# 2. Mount Google Drive and switch to the data folder.
import os
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
os.makedirs("02_pilot_scorer_family", exist_ok=True)
os.environ["HF_HOME"] = "/content/hf"

In [ ]:
# 3. Install packages.
%pip install -q -U transformers huggingface-hub accelerate

In [ ]:
# 4. Score each text with the model that wrote it: each generator scores only its own two conditions.
import gc, glob, json, shutil, traceback
import numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

MODELS = ["facebook/opt-2.7b", "Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-3B-Instruct"]

def score_texts(scorer_name, records):
    tok = AutoTokenizer.from_pretrained(scorer_name)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
    model = AutoModelForCausalLM.from_pretrained(scorer_name, dtype=dtype).cuda().eval()
    out = []
    with torch.no_grad():
        for i, r in enumerate(records):
            k = len(tok(r["prefix"])["input_ids"])
            for who in ("human", "ai"):
                ids = tok(r["prefix"] + r[who], truncation=True, max_length=512)["input_ids"]
                if len(ids) - k < 10:
                    continue
                x = torch.tensor([ids]).cuda()
                lp = torch.log_softmax(model(x).logits[0, :-1].float(), -1)
                p = lp.exp()
                e1 = (p * lp).sum(-1)
                rows = [-lp.gather(1, x[0, 1:, None])[:, 0], -e1, (p * lp * lp).sum(-1) - e1 ** 2,
                        p.topk(10, -1).values.sum(-1)]
                arr = torch.stack(rows).cpu().numpy().astype(np.float32)[:, k - 1:]
                out.append(dict(id=r["id"], domain=r["domain"], model=r["model"], decoding=r["decoding"],
                                scorer=scorer_name, who=who, arr=arr))
            if i % 50 == 0:
                print(" ", scorer_name, i, "/", len(records), flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()
    return out

for m in MODELS:
    path = f"02_pilot_scorer_family/scores_{m.split('/')[1]}.pkl"
    if os.path.exists(path):
        print("Already done, skipping:", path); continue
    try:
        recs = []
        for f in sorted(glob.glob(f"01_pilot_decoding/gen_{m.split('/')[1]}_*.json")):
            recs += json.load(open(f))
        assert recs, f"No continuation files found for {m}"
        pd.DataFrame(score_texts(m, recs)).to_pickle(path)
        print(m, "done:", len(recs) * 2, "texts")
    except Exception:
        traceback.print_exc(); print("This scorer failed, skipping:", m)
    finally:
        gc.collect(); torch.cuda.empty_cache()
        shutil.rmtree(f"/content/hf/models--{m.replace('/', '--')}", ignore_errors=True)

In [ ]:
# 5. Summary: the same texts scored by their own generator vs by GPT-2 XL.
from sklearn.metrics import roc_auc_score
s = pd.concat([pd.read_pickle(f) for f in sorted(glob.glob("02_pilot_scorer_family/scores_*.pkl"))], ignore_index=True)
s["fdg"] = [(a[1].sum() - a[0].sum()) / np.sqrt(a[2].sum()) for a in s.arr]
s["y"] = (s.who == "ai").astype(int)
matched = s.groupby(["model", "decoding"]).apply(lambda g: roc_auc_score(g.y, g.fdg)).rename("AUROC_self_scored")
old = pd.read_csv("01_pilot_decoding/summary.csv").set_index(["model", "decoding"])["Fast-DetectGPT AUROC"].rename("AUROC_GPT2XL_scored")
table = pd.concat([old, matched], axis=1)
table.to_csv("02_pilot_scorer_family/summary.csv")
print(table.round(3))

In [ ]:
# 6. Flush unsynced files back to Google Drive.
%cd /content
drive.flush_and_unmount()
print("Done: results are in DATA_DIR/02_pilot_scorer_family/")